# 01 — Generate the synthetic claims dataset

Runs all three stages of dataset construction (Section 3.2.1) on a Colab GPU and saves
everything to Google Drive:

| Stage | Module | Output (in Drive) |
|---|---|---|
| A. Structured claims | `src.data.generator` | `data/raw/<version>/base_claims.parquet`, `note_requests.jsonl` |
| B. LLM clinical notes | `src.data.note_synthesis` | `data/raw/<version>/notes.jsonl`, `boilerplate_notes.jsonl` |
| C. Injection, labels, splits | `src.data.build_dataset` | `data/processed/<version>/*.parquet`, `manifest.json` |

**Before running:** Runtime → Change runtime type → T4 GPU. Put a `.env` containing
`GITHUB_TOKEN=...` (a fine-grained, read-only token for `victorgpt0/claimguard`) at
`MyDrive/claimguard/.env`, or add it as a Colab secret; never paste it into a cell.

Stage B appends notes in batches, so if the runtime disconnects, rerun the notebook and it
resumes where it stopped.

In [ ]:
DATASET_VERSION = 'v0.4-5k'   # bump when the generator, catalogue or parameters change
N_CLAIMS = 5000
SEED = 42
BRANCH = 'dev'

## 1. Bootstrap: Drive, repository, run record

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, subprocess, sys
REPO_DIR = '/content/claimguard'

DRIVE_ENV = '/content/drive/MyDrive/claimguard/.env'

def load_env_file(path):
    # Minimal .env reader (KEY=value lines), so this cell needs no extra package.
    if not os.path.exists(path):
        return
    for line in open(path):
        line = line.strip()
        if line and not line.startswith('#') and '=' in line:
            key, value = line.split('=', 1)
            os.environ.setdefault(key.strip(), value.strip().strip('"\''))

def github_token():
    # The kernel runs on Colab's VM, so a local .env (VS Code's jupyter.envFile) never
    # reaches it. Read the .env kept in Drive, then a Colab secret, then prompt.
    load_env_file(DRIVE_ENV)
    if os.getenv('GITHUB_TOKEN'):
        return os.environ['GITHUB_TOKEN']
    try:
        from google.colab import userdata
        return userdata.get('GITHUB_TOKEN')
    except Exception:
        pass
    from getpass import getpass
    return getpass('GITHUB_TOKEN (read-only, not saved): ').strip()

if not os.path.exists(REPO_DIR):
    token = github_token()
    if not token:
        raise RuntimeError('No GITHUB_TOKEN: add a Colab secret or enter it at the prompt')
    subprocess.run(['git', 'clone', '-q', '-b', BRANCH,
                    f'https://{token}@github.com/victorgpt0/claimguard.git', REPO_DIR], check=True)
else:
    subprocess.run(['git', '-C', REPO_DIR, 'pull', '-q'], check=True)
os.chdir(REPO_DIR)
sys.path.insert(0, REPO_DIR)

COMMIT = subprocess.run(['git', 'rev-parse', 'HEAD'], capture_output=True, text=True).stdout.strip()
ROOT = '/content/drive/MyDrive/claimguard'
RAW = f'{ROOT}/data/raw/{DATASET_VERSION}'
PROCESSED = f'{ROOT}/data/processed/{DATASET_VERSION}'
ENV_LOGS = f'{ROOT}/artifacts/environment_logs'
for d in (RAW, PROCESSED, ENV_LOGS):
    os.makedirs(d, exist_ok=True)
print('commit', COMMIT)

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv
!python --version

## 2. Stage A: structured claims (CPU)

In [ ]:
!pip install -q pyarrow pyyaml
!python -m src.data.generator --n-claims {N_CLAIMS} --seed {SEED} --out {RAW}

## 3. Stage B: clinical notes (GPU)

Tries vLLM first (pinned to a release built for Colab's CUDA runtime). If the install or
start-up fails on the assigned GPU, set `BACKEND = 'transformers'`, which is slower
(about 1–2 h for 5,000 notes on a T4) but runs anywhere.

In [ ]:
BACKEND = 'vllm'   # or 'transformers'
if BACKEND == 'vllm':
    !pip install -q vllm==0.19.0

In [ ]:
!python -m src.data.note_synthesis --backend {BACKEND} --seed {SEED} \
    --requests {RAW}/note_requests.jsonl --out {RAW}/notes.jsonl \
    --boilerplate-out {RAW}/boilerplate_notes.jsonl

### Inspect before building

Read a sample of notes yourself. Record what you checked, and anything you rejected, in
`docs/ai_use_log.md`: the notes are AI-generated research data.

In [ ]:
import pandas as pd
from pathlib import Path   # a Path, so a missing file raises FileNotFoundError
notes = pd.read_json(Path(RAW, 'notes.jsonl'), lines=True)
boiler = pd.read_json(Path(RAW, 'boilerplate_notes.jsonl'), lines=True)
print(len(notes), 'notes;', notes['rejected_reason'].notna().sum(), 'rejected:')
print(notes['rejected_reason'].value_counts())
print(len(boiler), 'boilerplate;', boiler['rejected_reason'].notna().sum(), 'rejected')
requests = pd.read_json(Path(RAW, 'note_requests.jsonl'), lines=True)
sample = notes.merge(requests, on='claim_id').sample(8, random_state=SEED)
for row in sample.itertuples():
    print(f'\n[{row.icd11_code}] {row.diagnosis} | {row.patient_age}{row.patient_sex[0]} {row.package}')
    print(row.clinical_notes)

In [ ]:
print('Accepted boilerplate:')
for n in boiler.loc[boiler['rejected_reason'].isna(), 'clinical_notes'].sample(10, random_state=SEED):
    print(' -', n)

## 4. Stage C: inject fraud, label, split

In [ ]:
!python -m src.data.build_dataset --raw {RAW} --out {PROCESSED} --seed {SEED}

In [ ]:
import json, shutil, datetime
manifest = json.load(open(f'{PROCESSED}/manifest.json'))
manifest['colab_commit'] = COMMIT
json.dump(manifest, open(f'{PROCESSED}/manifest.json', 'w'), indent=2)
stamp = datetime.date.today().isoformat()
!pip freeze > {ENV_LOGS}/requirements_frozen_{stamp}_dataset.txt
pd.DataFrame(manifest['splits']).T[['claims', 'fraud', 'inconsistent', 'text_noise']]